# Guide de Référence - Troubleshooting et Cas d'Usage

Ce notebook documente les problèmes courants et leurs solutions

## 1. Problèmes Courants et Solutions

### ❌ ModuleNotFoundError: No module named 'yaml'
**Solution:**
```bash
pip install PyYAML
```

### ❌ ModuleNotFoundError: No module named 'evidently.report'
**Cause:** Version incompatible d'Evidently (< 0.8)
**Solution:** Le code gère automatiquement le fallback avec rapports HTML simples
```bash
pip install --upgrade evidently>=0.7.0
```

### ❌ ERROR: at least one array or dtype is required
**Cause:** Données malformées ou vides
**Solution:**
1. Vérifier le fichier CSV n'est pas vide
2. S'assurer que la dernière colonne est la target
3. Nettoyer les valeurs NaN

### ❌ FileNotFoundError: data/raw/...
**Cause:** Le fichier de données n'existe pas
**Solution:** Placer les fichiers CSV dans `data/raw/` avant d'exécuter

### ⚠️ MLflow connection refused
**Cause:** MLflow server n'est pas en fonctionnement
**Solution:**
```bash
mlflow ui  # Démarre le serveur sur http://localhost:5000
```
Ou configurer dans `utils/config.yaml`:
```yaml
mlflow:
  tracking_uri: "file:./mlruns"  # Local files instead of server
```

## 2. Checklist de Diagnostic

In [ ]:
import os
import sys
import pandas as pd

print("=== DIAGNOSTIC COMPLET ===")

# 1. Répertoires
print("\n1️⃣ RÉPERTOIRES:")
dirs_to_check = [
    ("data/raw", "Données brutes"),
    ("data/processed", "Données traitées"),
    ("Models", "Modèles sauvegardés"),
    ("outputs", "Rapports"),
    ("utils", "Scripts utils"),
    ("notebooks", "Notebooks"),
]

for dir_path, desc in dirs_to_check:
    exists = "✓" if os.path.exists(dir_path) else "✗"
    print(f"  {exists} {dir_path} ({desc})")

# 2. Fichiers essentiels
print("\n2️⃣ FICHIERS ESSENTIELS:")
files_to_check = [
    ("utils/config.yaml", "Configuration"),
    ("utils/data_loader.py", "Chargement données"),
    ("utils/model.py", "Entraînement modèle"),
    ("utils/data_validator.py", "Validation Evidently"),
    ("utils/performance_monitor.py", "Monitoring"),
    ("utils/pipeline.py", "Pipeline complet"),
]

for file_path, desc in files_to_check:
    exists = "✓" if os.path.exists(file_path) else "✗"
    print(f"  {exists} {file_path} ({desc})")

# 3. Fichiers données
print("\n3️⃣ DONNÉES CSV:")
raw_path = "data/raw/"
if os.path.exists(raw_path):
    csv_files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]
    if csv_files:
        for f in csv_files:
            path = os.path.join(raw_path, f)
            size_mb = os.path.getsize(path) / (1024*1024)
            try:
                df = pd.read_csv(path)
                print(f"  ✓ {f} ({df.shape[0]} lignes, {df.shape[1]} colonnes, {size_mb:.2f} MB)")
            except Exception as e:
                print(f"  ✗ {f} - Erreur: {e}")
    else:
        print(f"  ✗ Aucun fichier CSV trouvé dans {raw_path}")
else:
    print(f"  ✗ {raw_path} n'existe pas")

# 4. Dépendances
print("\n4️⃣ DÉPENDANCES PYTHON:")
packages = ['pandas', 'sklearn', 'mlflow', 'evidently', 'yaml']
for pkg in packages:
    try:
        __import__(pkg)
        print(f"  ✓ {pkg}")
    except ImportError:
        print(f"  ✗ {pkg} - À installer")

print("\n✓ Diagnostic complet")

## 3. Cas d'Usage: Classification vs Régression

In [ ]:
print("""
=== DÉTERMINER LE TYPE DE PROBLEME ===

1️⃣ CLASSIFICATION (sortie discrète):
   - Exemple: Prédire si un client va churn (Oui/Non)
   - Target: valeurs catégories (0, 1, 2, ...)
   - Heuristique utilisée: len(unique values) < 20
   - Métriques: Accuracy, Precision, Recall, F1-score
   - Modèle: RandomForestClassifier

2️⃣ RÉGRESSION (sortie continue):
   - Exemple: Prédire la consommation électrique
   - Target: values numériques continues
   - Heuristique utilisée: len(unique values) >= 20
   - Métriques: MSE, RMSE, R²
   - Modèle: RandomForestRegressor

⚠️  Si la détection automatique échoue, modifier:
   dans utils/model.py, fonction train_model(), ligne 29
""")

## 4. Exemple: Créer Données de Test

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("=== CRÉER DONNÉES DE TEST ===")

# 1. Classification - Dataset simple
print("\n1️⃣ Dataset Classification (Iris-like)...")
np.random.seed(42)
n_samples = 150

X = np.random.randn(n_samples, 4) * 2 + 5
y = np.random.choice([0, 1, 2], n_samples)

df_classification = pd.DataFrame(X, columns=['feature_1', 'feature_2', 'feature_3', 'feature_4'])
df_classification['target'] = y

Path("data/raw").mkdir(parents=True, exist_ok=True)
df_classification.to_csv("data/raw/classification_sample.csv", index=False)
print(f"  ✓ classification_sample.csv créé ({df_classification.shape})")
print(f"    Aperçu:")
print(df_classification.head())

# 2. Régression - Dataset simple
print("\n2️⃣ Dataset Régression (Énergie-like)...")
X_reg = np.random.randn(n_samples, 5) * 10 + 50
y_reg = X_reg.sum(axis=1) + np.random.randn(n_samples) * 5

df_regression = pd.DataFrame(X_reg, columns=['temp_matin', 'temp_midi', 'temp_soir', 'humidite', 'vent'])
df_regression['consommation_kwh'] = y_reg

df_regression.to_csv("data/raw/regression_sample.csv", index=False)
print(f"  ✓ regression_sample.csv créé ({df_regression.shape})")
print(f"    Aperçu:")
print(df_regression.head())

print("\n✓ Fichiers de test créés dans data/raw/")

## 5. Exemple: Pipeline Pas à Pas

In [ ]:
import sys
import os
sys.path.insert(0, os.getcwd())

from utils.data_loader import load_data, split_data
from utils.model import train_model, evaluate_model

print("=== PIPELINE SIMPLE PAS À PAS ===")

# Sélectionner un fichier de test
data_file = "data/raw/classification_sample.csv"

if os.path.exists(data_file):
    print(f"\n📂 Fichier: {data_file}")
    
    # ÉTAPE 1
    print("\n[1/4] Chargement des données...")
    data = load_data(data_file)
    
    # ÉTAPE 2
    print("\n[2/4] Préparation (train/test split)...")
    X_train, X_test, y_train, y_test = split_data(data, test_size=0.2)
    
    # ÉTAPE 3
    print("\n[3/4] Entraînement du modèle...")
    model = train_model(X_train, y_train)
    
    # ÉTAPE 4
    print("\n[4/4] Évaluation...")
    metrics = evaluate_model(model, X_test, y_test)
    
    if metrics:
        print("\n✓ Pipeline exécuté avec succès!")
        print(f"\nMétriques:")
        for key, value in metrics.items():
            print(f"  {key}: {value:.4f}")
    else:
        print("✗ Erreur lors de l'évaluation")
else:
    print(f"✗ Fichier non trouvé: {data_file}")

## 6. Configuration MLflow (Local)

In [ ]:
print("""
=== CONFIGURATION MLFLOW ===

1️⃣ OPTION LOCAL (Recommandée pour développement):

Modifier utils/config.yaml:
```yaml
mlflow:
  tracking_uri: "file:./mlruns"
  experiment_name: "enedis_project"
```

Avantages:
  - Pas de serveur à démarrer
  - Données locales
  - Idéal pour la formation

2️⃣ OPTION SERVER (Production):

Démarrer le serveur:
```bash
mlflow ui --host 0.0.0.0 --port 5000
```

Modifier config.yaml:
```yaml
mlflow:
  tracking_uri: "http://localhost:5000"
  experiment_name: "enedis_project"
```

Accès:
  - Interface: http://localhost:5000
  - API: http://localhost:5000/api/2.0/mlflow

3️⃣ VISUALISER LES RUNS:

Avec option LOCAL:
```bash
mlflow ui  # Va afficher le répertoire mlruns/
```

Vous verrez:
  - Toutes les runs
  - Paramètres et métriques
  - Modèles sauvegardés
  - Comparaisons entre runs
""")

## 7. Structure des Fichiers Générés

In [ ]:
import os
from pathlib import Path

print("""
=== STRUCTURE DES FICHIERS GENERÉS ===

Après exécution du pipeline:

```
workspace/
├── data/
│   ├── raw/                          # Données source
│   │   ├── extract_cvs_engis*.csv
│   │   └── classification_sample.csv (créé)
│   └── processed/                    # Données traitées
│       └── reference.csv (optionnel)
│
├── outputs/                          # Résultats
│   ├── evidently_validation.html     # Rapport validation
│   └── evidently_performance.html    # Rapport performance
│
├── Models/                           # Modèles (MLflow gère)
│   └── (vides, sauvegarde via MLflow)
│
├── mlruns/                           # MLflow local (si config local)
│   └── 0/                            # Expérience
│       └── xxxxx/                    # Run ID
│           ├── artifacts/
│           │   └── model/            # Modèle pickle
│           ├── metrics/              # Métriques JSON
│           └── params/               # Paramètres JSON
│
├── utils/
│   ├── config.yaml
│   ├── data_loader.py
│   ├── model.py
│   ├── data_validator.py
│   ├── performance_monitor.py
│   ├── mlflow_tracker.py
│   └── pipeline.py
│
└── notebooks/
    ├── 01_MLOps_Pipeline_Diagnostic.ipynb
    ├── 02_MLOps_Advanced_Evidently_MLflow.ipynb
    └── 03_Guide_Reference_Troubleshooting.ipynb  (ce fichier)
```
""")

# Afficher la structure actuelle
print("\n📊 STRUCTURE ACTUELLE:")
for root, dirs, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    
    # Limiter la profondeur pour lisibilité
    if level < 3:
        subindent = " " * 2 * (level + 1)
        for file in files[:5]:  # Max 5 fichiers par dossier
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... et {len(files)-5} fichiers")
    
    if level >= 2:  # Arrêter à profondeur 2
        dirs.clear()

## 8. Ressources et Liens

### 📚 Documentation
- [README_UTILS.md](../README_UTILS.md) - Documentation complète des modules
- [copilot-instructions.md](../../.github/copilot-instructions.md) - Conventions du projet

### 🔗 Liens externes
- [Scikit-learn Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#random-forests)
- [Evidently AI](https://www.evidentlyai.com/)
- [MLflow Documentation](https://mlflow.org/docs/latest/)
- [Pandas Documentation](https://pandas.pydata.org/docs/)

### 🎯 Prochaines étapes
1. ✓ Créer données de test (cf. notebook 3 Cas d'Usage)
2. Exécuter notebook 01 - Diagnostic
3. Exécuter notebook 02 - Advanced
4. Adapter les paramètres dans `utils/config.yaml`
5. Déployer en production

### 💡 Tips
- Tester d'abord avec des petits datasets
- Vérifier les colonnes du CSV (dernière = target)
- Monitorer les rapports Evidently HTML
- Sauvegarder mlruns/ en version control